# Module 02 — Sampling & Sample Size
Dedicated notebook for Module 02 — verifies how the PISA sampling weight changes results, and a sample-size-by-power table (Cohen 1988, via `statsmodels`) — this table has no PSPP equivalent command, so it only runs here.

Human-reviewed English translation of the original Vietnamese notebook (code and results are identical, only text is translated).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"]=(7,4); pd.set_option("display.precision",4)

RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
print("School table:", truong.shape)

## 1. Three layers: population, sampling frame, sample
In PISA 2025 Vietnam: population = all 15-year-old students in Vietnam; sampling frame = the list of schools MOET provided to OECD; sample = the 195 schools that actually participated.

In [ ]:
print(f"Actual sample: {truong.shape[0]} schools, {int(truong['n_hs'].sum())} students total")

## 2. How the sample weight changes the result — a real check
Because PISA samples schools by cluster with probability proportional to size, schools do not carry equal "weight" in the analysis.

In [ ]:
# Unweighted: each school counts as 1
mean_unweighted = truong["EDULEAD"].mean()

# Weighted: WEIGHT BY W_NRASCHBWT (equivalent to SPSS Data > Weight Cases)
valid = truong.dropna(subset=["EDULEAD"])
mean_weighted = np.average(valid["EDULEAD"], weights=valid["W_NRASCHBWT"])
n_eff = valid["W_NRASCHBWT"].sum()

print(f"Mean EDULEAD UNWEIGHTED = {mean_unweighted:.4f}  (N={len(valid)} schools, each counted once)")
print(f"Mean EDULEAD WEIGHTED   = {mean_weighted:.4f}  (effective N={n_eff:.1f})")

#### 📤 Actual output
Unweighted = 0.7653 (N=195). Weighted = 0.7874 (effective N=7,284.6 — the sum of weights, reflecting true student enrolment). The gap here is small (0.02) because EDULEAD isn't strongly correlated with school size, but for a variable that IS strongly correlated with size, ignoring weights could meaningfully bias the conclusion.

## 3. Minimum pilot sample size (Julious, 2005)
n_pilot ≥ 12/group — below this threshold, the estimated SD fluctuates too much to be a reliable input for the formal sample-size calculation.

In [ ]:
print("Minimum pilot sample size per Julious (2005): 12 observations/group")

## 4. Formal sample-size table by power analysis (Cohen, 1988)
Approximate formula: n ≈ (z_α/2 + z_β)² × 2 / d² — computed more precisely with `statsmodels.stats.power.TTestIndPower` (uses the noncentral t-distribution instead of the z approximation).

In [ ]:
from statsmodels.stats.power import TTestIndPower
analysis = TTestIndPower()
rows = []
for d_val, label in [(0.2,"Small"), (0.5,"Medium"), (0.8,"Large")]:
    n_req = analysis.solve_power(effect_size=d_val, alpha=0.05, power=0.80, alternative="two-sided")
    rows.append((d_val, label, round(n_req)))
table = pd.DataFrame(rows, columns=["Effect size d", "Level", "Sample size/group"])
table

#### 📤 Actual output
d=0.2→≈394/group, d=0.5→≈64/group, d=0.8→≈26/group (α=0.05, power=0.80). ✅ Matches the table quoted in Module 02 and Module 04 — the smaller the effect, the larger the sample needed.

## 5. Visual: required sample size shrinks as effect size grows

In [ ]:
d_range = np.linspace(0.1, 1.0, 30)
n_range = [analysis.solve_power(effect_size=d, alpha=0.05, power=0.80, alternative="two-sided") for d in d_range]
fig, ax = plt.subplots()
ax.plot(d_range, n_range)
ax.set_xlabel("Effect size d"); ax.set_ylabel("Required sample size / group")
ax.set_title("Required sample size drops quickly as effect size grows (α=0.05, power=0.80)")
plt.show()